In [1]:
import modelling_functions as mf
import numpy as np
import pyomo.environ as pyo

D-Wave Token unspecified. Can not choose any quantum solver


In [2]:
def generate_random_distance_matrix(num_points):
    # Generate a random distance matrix
    matrix = np.random.rand(num_points, num_points)
    # Ensure the matrix is symmetric and has zeros on the diagonal
    matrix = (matrix + matrix.T) / 2
    np.fill_diagonal(matrix, 0)
    return matrix

def make_QUBO_upper_triangle(Q):
    Q = np.array(Q)
    return(np.triu(Q) + np.tril(Q,-1).T)

def calculate_density(QUBO, tol=1e-8):
    values = QUBO[np.triu_indices(len(QUBO),1)]
    density = np.count_nonzero(abs(values) > tol) / len(values)
    return density

In [3]:
P = 5
T = 20
# arc flow:
m = pyo.ConcreteModel()
m.Nodes = pyo.RangeSet(P) * pyo.RangeSet(0, T-1) | pyo.Set(initialize=[(0,-1)]) | pyo.Set(initialize=[(0,T)])
m.Arcs = pyo.Set(initialize=lambda m: [(p,t,q,s) for p,t,q,s in m.Nodes*m.Nodes if t+1 == s])
m.x = pyo.Var(m.Arcs, within=pyo.Binary)
def flow_balance(m, p, t):
    following_nodes = [(q,s) for (q,s) in m.Nodes if s == t+1]
    precceding_nodes = [(q,s) for (q,s) in m.Nodes if s == t-1]
    if t == -1:
         right_side = 1
    elif t == T:
         right_side = -1   
    else:
         right_side = 0
    return sum(m.x[p,t,following_node] for following_node in following_nodes) - sum(m.x[precceding_node,p,t] for precceding_node in precceding_nodes) == right_side
        
m.flow_balance = pyo.Constraint(m.Nodes, rule = flow_balance)
distance_matrix = generate_random_distance_matrix(P)
def travel_cost(m,p,t,q,s):
     if p in [0, P+1] or q in [0, P+1]:
          return 0
     else:
          return distance_matrix[p-1,q-1]
m.cost = pyo.Param(m.Arcs, initialize=travel_cost)
m.obj = pyo.Objective(expr=sum(m.cost[arc] * m.x[arc] for arc in m.Arcs))
inst = m.create_instance()

In [20]:
n = pyo.ConcreteModel()
n.x = pyo.Var(within=pyo.NonNegativeReals)
n.obj = pyo.Objective(expr=n.x)
n.ineq = pyo.Constraint(expr=n.x <= 1)
ins = n.create_instance()
inst = mf.binary_discretization(ins, [0,2], precision=3, encoding='log', verbose=True)

3 (1 integer and 2 fractional) binary encoding variables added
1 variables fixed


In [21]:
inst.pprint()

2 Set Declarations
    x_fractional_encoding_variable_index : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {1, 2}
    x_integer_encoding_variable_index : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    1 :    {0,}

3 Var Declarations
    x : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :     0 :  None :  True : False : NonNegativeReals
    x_fractional_encoding_variable : Size=2, Index=x_fractional_encoding_variable_index
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :  None :     1 : False :  True : Binary
          2 :     0 :  None :     1 : False :  True : Binary
    x_integer_encoding_variable : Size=1, Index=x_integer_encoding_variable_index
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          0 :     0 :  None :     1 : Fa

In [22]:
from pyomo.core.expr.numeric_expr import decompose_term
constraint = inst.ineq
decomposed_body = decompose_term(constraint.body)[1]
a, x = list(zip(*decomposed_body))
min_a, max_a = sum(i for i in a if i < 0), sum(i for i in a if i > 0)
u, l = constraint.upper, constraint.lower

In [5]:
QUBO_pyomo = mf.BLP_to_QUBO(inst, penalties=1, slack_mode='log', verbose=True)
variables, QUBO, C = mf.pyomo_QUBO_to_array(QUBO_pyomo)
print('Size:', len(QUBO))
print('Density:', calculate_density(QUBO))

constraint of other form: ineq


ValueError: not enough values to unpack (expected 2, got 1)

In [80]:
QUBO

array([[  1.,   4.,   8.,   6.,  -2.,  -4.,  -8., -16., -10.],
       [  0.,   4.,  16.,  12.,  -4.,  -8., -16., -32., -20.],
       [  0.,   0.,  16.,  24.,  -8., -16., -32., -64., -40.],
       [  0.,   0.,   0.,   9.,  -6., -12., -24., -48., -30.],
       [  0.,   0.,   0.,   0.,   2.,   4.,   8.,  16.,  10.],
       [  0.,   0.,   0.,   0.,   0.,   6.,  16.,  32.,  20.],
       [  0.,   0.,   0.,   0.,   0.,   0.,  20.,  64.,  40.],
       [  0.,   0.,   0.,   0.,   0.,   0.,   0.,  72.,  80.],
       [  0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.,  30.]])